In [1]:
from tqdm import tqdm
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
from models_ import *
from utils import *

from flow_model import *

from flow_utils import *

from forward_inverse import *
import torch.nn.functional as F

# init

In [2]:
device = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')

In [3]:
device

device(type='mps')

In [4]:
def set_seed(seed: int = 42):
    random.seed(seed)                       
    np.random.seed(seed)                   
    torch.manual_seed(seed)                

    if torch.backends.mps.is_available():
  
        print("Using MPS: Seed fixed for reproducibility")
    elif torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False   

In [5]:
set_seed(42)

Using MPS: Seed fixed for reproducibility


# classfier expr

In [ ]:
models={
    
    "PCAMLP": FeatureEncoder(
        extractor=PCAMLP(d_in=32*32, d_out=16, d_hidden=64, img_size=32, n_components=256),
        selector=TopKSelector(d_model=16, k=8)),
     "MiniViT": FeatureEncoder(
        extractor=MiniVisionTransformer(d_in=1, d_out=16, d_hidden=32, patch_size=4, num_heads=4, img_size=32),
        selector=TopKSelector(d_model=16, k=8)),
    "MiniUNet": FeatureEncoder(
        extractor=MiniUNet(d_in=1, d_out=12, d_hidden=28,kernel_num=3, img_size=32),
        selector=TopKSelector(d_model=12, k=8)),
   
    
    "MLP": FeatureEncoder(
        extractor=MLPExtractor(d_in=32*32, d_out=10, d_hidden=17, img_size=32),
        selector=TopKSelector(d_model=10, k=8)),    
  
    
}
for key,values in models.items():
    print(f"{key} model size: {model_size_b(values) / MiB:.2f} MiB")
    print(f"{key} model params: {count_model_params(values)}")

In [ ]:
transform = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.ToTensor()
])

full_dataset = datasets.MNIST(root="./data", train=True, download=False, transform=transform)

In [ ]:
runner = ExperimentRunner(
    models=models,
    dataset=full_dataset,
    num_folds=5,
    num_epochs=5,
    batch_size=64,
    lr=1e-3,
    classfier_in=8
)


In [ ]:
classifier = UniversalClassifier(d_in=8, d_out=10).to(device)

In [ ]:
runner.run()

# flow matching test

## expr without selector

In [ ]:
path = GaussianConditionalProbabilityPath(
    p_data = MNISTSampler(),
    p_simple_shape = [1, 32, 32],
    alpha = LinearAlpha(),
    beta = LinearBeta()
).to(device)

In [ ]:
models = {
    "MiniViT": VecField(
        matcher=Matcher(
            d_in=16,
            latent_model=MiniVisionTransformer(
                    d_in=1, d_out=16, d_hidden=32, patch_size=4, num_heads=4, img_size=32
            ),
            out_channels=1
        )
    ),
    "MiniUNet": VecField(
        matcher=Matcher(
            d_in=16,
            latent_model=MiniUNet(
                    d_in=1, d_out=16, d_hidden=27, kernel_num=3, img_size=32
                ),
            out_channels=1
        )
    ),
    "PCAMLP": VecField(
        matcher=Matcher(
            d_in=16,
            latent_model=PCAMLP(
                    d_in=32*32, d_out=16, d_hidden=60, img_size=32, n_components=256
                ),
            out_channels=1
        )
    ),
    "MLP": VecField(
        matcher=Matcher(
            d_in=14,
            latent_model=MLPExtractor(
                    d_in=32*32, d_out=14, d_hidden=17, img_size=32
                ),
            out_channels=1
        )
    ),
}
for key,values in models.items():
    print(f"{key} model size: {model_size_b(values) / MiB:.2f} MiB")
    print(f"{key} model params: {count_model_params(values)}")

In [ ]:
runner = FlowExperimentRunner(
    models=models,
    cfg_class=CFG,       
    path=path,
    eta=0.1,
    num_epochs=2500,
    batch_size=250,
    lr=1e-3,
    device=device
    
)
runner.run()

# flow expr

In [6]:
path = GaussianConditionalProbabilityPath(
    p_data = MNISTSampler(),
    p_simple_shape = [1, 32, 32],
    alpha = LinearAlpha(),
    beta = LinearBeta()
).to(device)

## expr-origin

In [7]:
modelo = {
    "MiniViT": VecField(
        matcher=Matcher(
            d_in=8,
            latent_model=FeatureEncoder(
                extractor=MiniVisionTransformer(
                    d_in=1, d_out=16, d_hidden=32, patch_size=4, num_heads=4, img_size=32
                ),
                selector=TopKSelector(d_model=16, k=8)
            ),
            out_channels=1
        )
    ),
    "MiniUNet": VecField(
        matcher=Matcher(
            d_in=8,
            latent_model=FeatureEncoder(
                extractor=MiniUNet(
                    d_in=1, d_out=16, d_hidden=26, kernel_num=3, img_size=32
                ),
                selector=TopKSelector(d_model=16, k=8)  #
            ),
            out_channels=1
        )
    ),
    "PCAMLP": VecField(
        matcher=Matcher(
            d_in=8,
            latent_model=FeatureEncoder(
                extractor=PCAMLP(
                    d_in=32*32, d_out=16, d_hidden=57, img_size=32, n_components=256
                ),
                selector=TopKSelector(d_model=16, k=8)
            ),
            out_channels=1
        )
    ),
    "MLP": VecField(
        matcher=Matcher(
            d_in=8,
            latent_model=FeatureEncoder(
                extractor=MLPExtractor(
                    d_in=32*32, d_out=14, d_hidden=16, img_size=32
                ),
                selector=TopKSelector(d_model=14, k=8)
            ),
            out_channels=1
        )
    ),
}

In [8]:
for key,values in modelo.items():
    print(f"{key} model size: {model_size_b(values) / MiB:.2f} MiB")
    print(f"{key} model params: {count_model_params(values)}")

MiniViT model size: 0.10 MiB
MiniViT model params: 25943
MiniUNet model size: 0.10 MiB
MiniUNet model params: 26515
PCAMLP model size: 1.10 MiB
PCAMLP model params: 25696
MLP model size: 0.10 MiB
MLP model params: 26493


In [9]:
runner = FlowExperimentRunner(
    models=modelo,
    cfg_class=CFG,       
    path=path,
    eta=0.1,
    num_epochs=2500,
    batch_size=250,
    lr=1e-3,
    device=device
    
)
runner.run()


>>> Training Model: MiniViT


Epoch 2499, loss: 1162.482: : 2500it [05:18,  7.85it/s, train=1162.4816, val=1160.8043]



>>> Training Model: MiniUNet


Epoch 2499, loss: 1192.466: : 2500it [07:30,  5.54it/s, train=1192.4658, val=1198.2717]



>>> Training Model: PCAMLP


0it [00:00, ?it/s]/Users/aaaa/Downloads/modular_nn_experiment/models_.py:341: UserWarning: The operator 'aten::linalg_svd' is not currently supported on the MPS backend and will fall back to run on the CPU. This may have performance implications. (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/aten/src/ATen/mps/MPSFallback.mm:14.)
  U, S, Vh = torch.linalg.svd(x_centered, full_matrices=False)
Epoch 2499, loss: 1200.594: : 2500it [03:27, 12.07it/s, train=1200.5938, val=1191.4207]



>>> Training Model: MLP


Epoch 2499, loss: 1235.295: : 2500it [02:49, 14.78it/s, train=1235.2947, val=1237.5771]


In [10]:
from torch.nn.functional import pairwise_distance,cosine_similarity

In [11]:
z, y = path.p_data.sample(1000) 
t = torch.rand(1000,1,1,1).to(z)
x = path.sample_conditional_path(z,t) 
ut_ref = path.conditional_vector_field(x,z,t) 

In [12]:
for key, value in modelo.items():
    
    # Forward and inverse outputs
    ut_theta = value(x, t=t, y=y)
    print(f"Model: {key}")
    
    # Loss and MSE
    loss = einsum(torch.square(ut_theta - ut_ref), 'b  c h w -> b').mean()
    mse = F.mse_loss(ut_theta ,ut_ref)
    print(f"  Loss: {loss}")
    print(f"  MSE: {mse}")
    cos_sim = cosine_similarity(rearrange(ut_theta, 'b c h w-> b (c h w)'), rearrange(ut_ref, 'b c h w -> b (c h w)'))
    print(f"  Cosine Similarity: {cos_sim.mean()}")

Model: MiniViT
  Loss: 1156.4698486328125
  MSE: 1.129365086555481
  Cosine Similarity: 0.6308457255363464
Model: MiniUNet
  Loss: 1194.8125
  MSE: 1.16680908203125
  Cosine Similarity: 0.6148016452789307
Model: PCAMLP
  Loss: 1188.256103515625
  MSE: 1.160406470298767
  Cosine Similarity: 0.6177289485931396
Model: MLP
  Loss: 1237.4263916015625
  MSE: 1.2084242105484009
  Cosine Similarity: 0.5970396995544434


## expr-1 forward path end2end

In [13]:
models_full={ "MiniViT":
    vecfield( LatentVecField(MiniVisionTransformer(
                    d_in=1, d_out=16, d_hidden=32, patch_size=4, num_heads=4, img_size=32),16),Utdecoder(16,1)),
    "MiniUNet":
    vecfield( LatentVecField(MiniUNet(
                    d_in=1, d_out=16, d_hidden=26, kernel_num=3, img_size=32
                ),16),Utdecoder(16,1)),
    "PCAMLP":
    vecfield(LatentVecField(PCAMLP(
                    d_in=32*32, d_out=16, d_hidden=57, img_size=32, n_components=256
                ), 16),Utdecoder(16,1)),
    "MLP":
    vecfield( LatentVecField(MLPExtractor(
                    d_in=32*32, d_out=14, d_hidden=16, img_size=32
                ), 14),Utdecoder(14,1)),
    

   }

In [14]:
for key,values in models_full.items():
    print(f"{key} model size: {model_size_b(values) / MiB:.2f} MiB")
    print(f"{key} model params: {count_model_params(values)}")

MiniViT model size: 0.13 MiB
MiniViT model params: 33187
MiniUNet model size: 0.13 MiB
MiniUNet model params: 33759
PCAMLP model size: 1.13 MiB
PCAMLP model params: 32940
MLP model size: 0.12 MiB
MLP model params: 31907


In [15]:
runner = FlowExperimentRunner(
    models=models_full,
    cfg_class=CFG,       
    path=path,
    eta=0.1,
    num_epochs=2500,
    batch_size=250,
    lr=1e-3,
    device=device
    
)
runner.run()


>>> Training Model: MiniViT


Epoch 2499, loss: 1200.433: : 2500it [04:54,  8.49it/s, train=1200.4326, val=1200.4845]



>>> Training Model: MiniUNet


Epoch 2499, loss: 1195.225: : 2500it [05:07,  8.14it/s, train=1195.2253, val=1211.5173]



>>> Training Model: PCAMLP


Epoch 2499, loss: 1160.684: : 2500it [03:07, 13.33it/s, train=1160.6844, val=1172.1993]



>>> Training Model: MLP


Epoch 2499, loss: 1165.393: : 2500it [02:57, 14.10it/s, train=1165.3931, val=1165.7828]


## expr-2 latent ->inverse

In [16]:
latentmodels = {
    "MiniViT": {
        "forward": LatentVecField(MiniVisionTransformer(
                    d_in=1, d_out=16, d_hidden=32, patch_size=4, num_heads=4, img_size=32),16),
        "inverse": MiniVisionTransformer(
                    d_in=1, d_out=16, d_hidden=32, patch_size=4, num_heads=4, img_size=32)
    },
    "ConvNet": {
        "forward": LatentVecField(MiniUNet(
                    d_in=1, d_out=16, d_hidden=26, kernel_num=3, img_size=32
                ),16),
        "inverse": MiniUNet(
                    d_in=1, d_out=16, d_hidden=26, kernel_num=3, img_size=32
                ),
    },
    "PCA_MLP": {
        "forward": LatentVecField(PCAMLP(
                    d_in=32*32, d_out=16, d_hidden=57, img_size=32, n_components=256
                ), 16),
        "inverse": PCAMLP(
                    d_in=32*32, d_out=16, d_hidden=58, img_size=32, n_components=256
                ),
    },
    "MLP": {
        "forward": LatentVecField(MLPExtractor(
                    d_in=32*32, d_out=14, d_hidden=16, img_size=32
                ), 14),
        "inverse": MLPExtractor(
                    d_in=32*32, d_out=14, d_hidden=16, img_size=32
                )
    }
}

In [17]:
for key, models_pair in latentmodels.items():
    forward_model = models_pair["forward"]
    inverse_model = models_pair["inverse"]
    
    print(f"\n[{key}]")
    
    print(f"  Forward model size: {model_size_b(forward_model) / MiB:.2f} MiB")
    print(f"  Forward model params: {count_model_params(forward_model)}")
    
    print(f"  Inverse model size: {model_size_b(inverse_model) / MiB:.2f} MiB")
    print(f"  Inverse model params: {count_model_params(inverse_model)}")
    
    total_params = count_model_params(forward_model) + count_model_params(inverse_model)
    total_size=(model_size_b(forward_model)+model_size_b(inverse_model))/ MiB
    print(f"  Total combined params: {total_params}")
    print(f"  Total combined size: {total_size:.2f} MiB")


[MiniViT]
  Forward model size: 0.06 MiB
  Forward model params: 16008
  Inverse model size: 0.06 MiB
  Inverse model params: 15824
  Total combined params: 31832
  Total combined size: 0.13 MiB

[ConvNet]
  Forward model size: 0.06 MiB
  Forward model params: 16580
  Inverse model size: 0.06 MiB
  Inverse model params: 16396
  Total combined params: 32976
  Total combined size: 0.13 MiB

[PCA_MLP]
  Forward model size: 1.06 MiB
  Forward model params: 15761
  Inverse model size: 1.06 MiB
  Inverse model params: 15850
  Total combined params: 31611
  Total combined size: 2.13 MiB

[MLP]
  Forward model size: 0.06 MiB
  Forward model params: 16799
  Inverse model size: 0.06 MiB
  Inverse model params: 16638
  Total combined params: 33437
  Total combined size: 0.13 MiB


In [18]:
runner = LatentFlowExperimentRunner(
    models=latentmodels,
    cfg_class=LatentCFG,
    path=path,           
    num_epochs=5000,
    batch_size=128,
    device=device,
    eta=0.1,
    lr=1e-3
)

runner.run()


>>> Training Model: MiniViT


Epoch 4999, loss: 0.223: : 5000it [07:45, 10.75it/s, train=0.2231, val=0.2313]     



>>> Training Model: ConvNet


Epoch 4999, loss: 0.093: : 5000it [07:32, 11.04it/s, train=0.0933, val=0.0950]     



>>> Training Model: PCA_MLP


Epoch 4999, loss: 0.001: : 5000it [02:18, 36.09it/s, train=0.0013, val=0.0010]



>>> Training Model: MLP


Epoch 4999, loss: 0.000: : 5000it [02:15, 36.80it/s, train=0.0001, val=0.0003]


In [19]:
z, y = path.p_data.sample(1000) 
t = torch.rand(1000,1,1,1).to(z)
x = path.sample_conditional_path(z,t) 
ut_ref = path.conditional_vector_field(x,z,t) 

In [20]:
from torch.nn.functional import pairwise_distance,cosine_similarity

In [21]:
for key, models_pair in latentmodels.items():
    forward_model = models_pair["forward"]
    inverse_model = models_pair["inverse"]

    # Forward and inverse outputs
    ut_forward = forward_model(x, t, y)
    ut_inverse = inverse_model(x)

    print(f"Model: {key}")
    
    # Loss and MSE
    loss = einsum(torch.square(ut_forward - ut_inverse), 'b seq d -> b').mean()
    mse = F.mse_loss(ut_forward, ut_inverse)
    print(f"  Loss: {loss}")
    print(f"  MSE: {mse}")

    # Reshape for probability-based metrics
    ut_forward = rearrange(ut_forward, 'b seq d -> b (seq d)')
    ut_inverse = rearrange(ut_inverse, 'b seq d -> b (seq d)')
    ut_forward_prob = torch.softmax(ut_forward, dim=-1)
    ut_inverse_prob = torch.softmax(ut_inverse, dim=-1)

    # KL Divergence
    kl_div_forward = torch.sum(ut_forward_prob * torch.log(ut_forward_prob / (ut_inverse_prob + 1e-8)), dim=-1)
    kl_div_inverse = torch.sum(ut_inverse_prob * torch.log(ut_inverse_prob / (ut_forward_prob + 1e-8)), dim=-1)
    print(f"  KL Divergence (forward): {kl_div_forward.mean()}")
    print(f"  KL Divergence (inverse): {kl_div_inverse.mean()}")

    # Wasserstein Distance
    wasserstein_distance = pairwise_distance(ut_forward, ut_inverse, p=2).mean()
    print(f"  Wasserstein Distance: {wasserstein_distance}")

    # JS Divergence
    m = 0.5 * (ut_forward_prob + ut_inverse_prob)
    kl_forward = torch.sum(ut_forward_prob * torch.log(ut_forward_prob / (m + 1e-8)), dim=-1)
    kl_inverse = torch.sum(ut_inverse_prob * torch.log(ut_inverse_prob / (m + 1e-8)), dim=-1)
    js_divergence = 0.5 * (kl_forward + kl_inverse)
    print(f"  JS Divergence: {js_divergence.mean()}")

    # Cosine Similarity
    cos_sim = cosine_similarity(ut_forward, ut_inverse, dim=-1)
    print(f"  Cosine Similarity: {cos_sim.mean()}")

    # Bhattacharyya Distance
    bc_coefficient = torch.sum(torch.sqrt(ut_forward_prob * ut_inverse_prob), dim=-1)
    bhattacharyya_distance = -torch.log(bc_coefficient + 1e-8)
    print(f"  Bhattacharyya Distance: {bhattacharyya_distance.mean()}")
    

Model: MiniViT
  Loss: 1.1641943454742432
  MSE: 0.0011194178368896246
  KL Divergence (forward): 0.000546827563084662
  KL Divergence (inverse): 0.0005477318190969527
  Wasserstein Distance: 0.9895530343055725
  JS Divergence: 0.00012895102554466575
  Cosine Similarity: -0.0014286234509199858
  Bhattacharyya Distance: 0.00013941216457169503
Model: ConvNet
  Loss: 0.05356411263346672
  MSE: 1.3077177754894365e-05
  KL Divergence (forward): -3.4451641113264486e-05
  KL Divergence (inverse): -3.445571201154962e-05
  Wasserstein Distance: 0.2101396769285202
  JS Divergence: -3.93465998058673e-05
  Cosine Similarity: 0.013067618943750858
  Bhattacharyya Distance: 1.6350173837054172e-06
Model: PCA_MLP
  Loss: 0.000984224141575396
  MSE: 6.151401612441987e-05
  KL Divergence (forward): 2.871531069104094e-05
  KL Divergence (inverse): 2.8648224542848766e-05
  Wasserstein Distance: 0.019876817241311073
  JS Divergence: 7.06097080183099e-06
  Cosine Similarity: 0.0031902736518532038
  Bhattacha

## expr-3 inverse-decoder

In [22]:
inversemodels = {
    "MiniViT": {
        "decoder": Utdecoder(16,1),
        "inverse": latentmodels['MiniViT']['inverse']
    },
    "ConvNet": {
        "decoder":  Utdecoder(16,1),
        "inverse": latentmodels['ConvNet']['inverse']
    },
    "PCA_MLP": {
        "decoder": Utdecoder(16,1),
        "inverse": latentmodels['PCA_MLP']['inverse']
    },
    "MLP": {
        "decoder":Utdecoder(14,1),
        "inverse": latentmodels['MLP']['inverse']
    }
}

In [23]:
for key, models_pair in inversemodels.items():
    decoder_model = models_pair["decoder"]
    inverse_model = models_pair["inverse"]
    
    print(f"\n[{key}]")
    
    print(f"  Decoder model size: {model_size_b(decoder_model) / MiB:.2f} MiB")
    print(f"  Decoder model params: {count_model_params(decoder_model)}")
    
    print(f"  Inverse model size: {model_size_b(inverse_model) / MiB:.2f} MiB")
    print(f"  Inverse model params: {count_model_params(inverse_model)}")
    
    total_params = count_model_params(decoder_model) + count_model_params(inverse_model)
    total_size=(model_size_b(decoder_model)+model_size_b(inverse_model))/ MiB
    print(f"  Total combined params: {total_params}")
    print(f"  Total combined size: {total_size:.2f} MiB")


    
    


[MiniViT]
  Decoder model size: 0.07 MiB
  Decoder model params: 17179
  Inverse model size: 0.06 MiB
  Inverse model params: 15824
  Total combined params: 33003
  Total combined size: 0.13 MiB

[ConvNet]
  Decoder model size: 0.07 MiB
  Decoder model params: 17179
  Inverse model size: 0.06 MiB
  Inverse model params: 16396
  Total combined params: 33575
  Total combined size: 0.13 MiB

[PCA_MLP]
  Decoder model size: 0.07 MiB
  Decoder model params: 17179
  Inverse model size: 1.06 MiB
  Inverse model params: 15850
  Total combined params: 33029
  Total combined size: 1.13 MiB

[MLP]
  Decoder model size: 0.06 MiB
  Decoder model params: 15108
  Inverse model size: 0.06 MiB
  Inverse model params: 16638
  Total combined params: 31746
  Total combined size: 0.12 MiB


In [24]:
runner = InverseFlowExperimentRunner(
  models=inversemodels,
    cfg_class=InverseutCFG,
    path=path,           
    num_epochs=5000,
    batch_size=128,
    device=device,
    eta=0.1,
    lr=1e-3)
runner.run()


>>> Training Model: MiniViT


Epoch 4999, loss: 1306.139: : 5000it [09:34,  8.70it/s, train=1306.1392, val=1305.3218]



>>> Training Model: ConvNet


Epoch 4999, loss: 1348.113: : 5000it [07:44, 10.76it/s, train=1348.1133, val=1348.1970]



>>> Training Model: PCA_MLP


Epoch 4999, loss: 1344.742: : 5000it [04:16, 19.51it/s, train=1344.7424, val=1347.5632]



>>> Training Model: MLP


Epoch 4999, loss: 1351.173: : 5000it [04:18, 19.34it/s, train=1351.1733, val=1366.2598]


In [25]:
for key, models_pair in inversemodels.items():
    decoder_model = models_pair["decoder"]
    inverse_model = models_pair["inverse"]
    ut_inverse = inverse_model(ut_ref)
    ut_theta_i=decoder_model(ut_inverse,t=t,y=y)
    loss2=einsum(torch.square(  ut_theta_i - ut_ref), 'b c h w -> b').mean()
    #loss3=einsum(torch.square(  ut_theta_i - ut_theta), 'b c h w -> b').mean()
    print(f"  Loss inverse-decoder: {loss2}")
   # print(f"  Loss forward-inverse: {loss3}")
    cos_sim = cosine_similarity(
        rearrange(ut_theta_i, 'b c h w -> b (c h w)'),
        rearrange(ut_ref, 'b c h w -> b (c h w)')
    )
    print(f"  Cosine Similarity: {cos_sim.mean()}")

  Loss inverse-decoder: 1298.040283203125
  Cosine Similarity: 0.5718196034431458
  Loss inverse-decoder: 1346.950439453125
  Cosine Similarity: 0.5492567420005798
  Loss inverse-decoder: 1347.9700927734375
  Cosine Similarity: 0.5493729710578918
  Loss inverse-decoder: 1365.82958984375
  Cosine Similarity: 0.5403342247009277


## expr-4 combnine encoder decoder 

In [26]:
mms = {
    "MiniViT":  vecfield(latentmodels['MiniViT']['forward'],inversemodels['MiniViT']['decoder']),
    
    "MiniUNet":vecfield(latentmodels['ConvNet']['forward'],inversemodels['ConvNet']['decoder']),
    "PCAMLP": vecfield(latentmodels['PCA_MLP']['forward'],inversemodels['PCA_MLP']['decoder']),
    "MLP":vecfield(latentmodels['MLP']['forward'],inversemodels['MLP']['decoder']),
}

In [27]:
for key, value in mms.items():
    
    # Forward and inverse outputs
    ut_theta = value(x, t=t, y=y)
    print(f"Model: {key}")
    
    # Loss and MSE
    loss = einsum(torch.square(ut_theta - ut_ref), 'b  c h w -> b').mean()
    mse = F.mse_loss(ut_theta ,ut_ref)
    print(f"  Loss: {loss}")
    print(f"  MSE: {mse}")
    cos_sim = cosine_similarity(rearrange(ut_theta, 'b c h w-> b (c h w)'), rearrange(ut_ref, 'b c h w -> b (c h w)'))
    print(f"  Cosine Similarity: {cos_sim.mean()}")

Model: MiniViT
  Loss: 1436.9066162109375
  MSE: 1.4032291173934937
  Cosine Similarity: 0.5093139410018921
Model: MiniUNet
  Loss: 1363.633544921875
  MSE: 1.3316733837127686
  Cosine Similarity: 0.5419068932533264
Model: PCAMLP
  Loss: 1395.967041015625
  MSE: 1.3632490634918213
  Cosine Similarity: 0.5290635228157043
Model: MLP
  Loss: 1365.8330078125
  MSE: 1.3338212966918945
  Cosine Similarity: 0.5403326749801636


In [28]:
z, y = path.p_data.sample(1000) 
t = torch.rand(1000,1,1,1).to(z)
x = path.sample_conditional_path(z,t)
ut_ref = path.conditional_vector_field(x,z,t) 

In [30]:
runner = FlowExperimentRunner(
    models=mms,
    cfg_class=CFG,       
    path=path,
    eta=0.1,
    num_epochs=2500,
    batch_size=250,
    lr=1e-3,
    device=device
    
)
runner.run()


>>> Training Model: MiniViT


Epoch 2499, loss: 1247.480: : 2500it [05:50,  7.14it/s, train=1247.4796, val=1242.9097]



>>> Training Model: MiniUNet


Epoch 2499, loss: 1230.455: : 2500it [05:03,  8.23it/s, train=1230.4553, val=1233.8972]



>>> Training Model: PCAMLP


Epoch 2499, loss: 1294.057: : 2500it [03:12, 13.02it/s, train=1294.0575, val=1296.8301]



>>> Training Model: MLP


Epoch 2499, loss: 1197.995: : 2500it [02:58, 14.00it/s, train=1197.9948, val=1203.1136]


In [31]:
for key, value in mms.items():
    
    # Forward and inverse outputs
    ut_theta = value(x, t=t, y=y)
    print(f"Model: {key}")
    
    # Loss and MSE
    loss = einsum(torch.square(ut_theta - ut_ref), 'b  c h w -> b').mean()
    mse = F.mse_loss(ut_theta ,ut_ref)
    print(f"  Loss: {loss}")
    print(f"  MSE: {mse}")
    cos_sim = cosine_similarity(rearrange(ut_theta, 'b c h w-> b (c h w)'), rearrange(ut_ref, 'b c h w -> b (c h w)'))
    print(f"  Cosine Similarity: {cos_sim.mean()}")

Model: MiniViT
  Loss: 1252.1339111328125
  MSE: 1.2227870225906372
  Cosine Similarity: 0.5938685536384583
Model: MiniUNet
  Loss: 1236.23828125
  MSE: 1.2072639465332031
  Cosine Similarity: 0.6003870964050293
Model: PCAMLP
  Loss: 1287.559326171875
  MSE: 1.2573823928833008
  Cosine Similarity: 0.5776249170303345
Model: MLP
  Loss: 1196.789306640625
  MSE: 1.168739676475525
  Cosine Similarity: 0.6155372858047485


In [32]:
cms = {
    "MiniViT": {
        "forward":latentmodels['MiniViT']['forward'] ,
        "inverse": latentmodels['MiniViT']['inverse'],
        "decoder":inversemodels['MiniViT']['decoder'],
    },
    "UNet": {
        "forward": latentmodels['ConvNet']['forward'],
        "inverse":latentmodels['ConvNet']['inverse'],
         "decoder":inversemodels['ConvNet']['decoder'],
    },
    "PCA_MLP": {
         "forward": latentmodels['PCA_MLP']['forward'],
        "inverse":latentmodels['PCA_MLP']['inverse'],
         "decoder":inversemodels['PCA_MLP']['decoder'],
    },
    "MLP": {
        "forward": latentmodels['MLP']['forward'],
        "inverse":latentmodels['MLP']['inverse'],
         "decoder":inversemodels['MLP']['decoder'],
    }
}

In [33]:
for key, value in cms.items():
    forward_model = value["forward"]
    inverse_model = value["inverse"]
    decoder_model = value["decoder"]
    
    # Forward and inverse outputs
    ut_forward = forward_model(x, t=t, y=y)
    ut_inverse = inverse_model(ut_ref)
    ut_theta = decoder_model(ut_forward,t=t,y=y)
    ut_theta_i=decoder_model(ut_inverse,t=t,y=y)
    
    print(f"Model: {key}")
    
    # Loss and MSE
    loss = einsum(torch.square(ut_theta - ut_ref), 'b c h w -> b').mean()
    loss2=einsum(torch.square(  ut_theta_i - ut_ref), 'b c h w -> b').mean()
    loss3=einsum(torch.square(  ut_theta_i - ut_theta), 'b c h w -> b').mean()
    mse = F.mse_loss(ut_theta, ut_ref)
    print(f"  Loss forward-decoder: {loss}")
    print(f"  Loss inverse-decoder: {loss2}")
    print(f"  Loss forward-inverse: {loss3}")
    print(f"  MSE: {mse}")
    
    cos_sim = cosine_similarity(
        rearrange(ut_theta, 'b c h w -> b (c h w)'),
        rearrange(ut_ref, 'b c h w -> b (c h w)')
    )
    print(f"  Cosine Similarity: {cos_sim.mean()}")

Model: MiniViT
  Loss forward-decoder: 1252.1339111328125
  Loss inverse-decoder: 1400.434814453125
  Loss forward-inverse: 151.79537963867188
  MSE: 1.2227870225906372
  Cosine Similarity: 0.5938685536384583
Model: UNet
  Loss forward-decoder: 1236.23828125
  Loss inverse-decoder: 1419.7119140625
  Loss forward-inverse: 180.42398071289062
  MSE: 1.2072639465332031
  Cosine Similarity: 0.6003870964050293
Model: PCA_MLP
  Loss forward-decoder: 1287.559326171875
  Loss inverse-decoder: 1379.6705322265625
  Loss forward-inverse: 65.67157745361328
  MSE: 1.2573823928833008
  Cosine Similarity: 0.5776249170303345
Model: MLP
  Loss forward-decoder: 1196.789306640625
  Loss inverse-decoder: 1410.58251953125
  Loss forward-inverse: 217.59552001953125
  MSE: 1.168739676475525
  Cosine Similarity: 0.6155372858047485


In [34]:
runner = FullFlowExperimentRunner(
    models=cms,
    cfg_class=FullCFG,
    path=path,           
    num_epochs=2500,
    batch_size=128,
    device=device,
    eta=0.1,
    lr=1e-3
)

runner.run()


>>> Training Model: MiniViT


Epoch 2499, loss: 478.358: : 2500it [05:24,  7.72it/s, train=478.3580, val=475.8911]



>>> Training Model: UNet


Epoch 2499, loss: 428.464: : 2500it [05:36,  7.43it/s, train=428.4643, val=434.4594]



>>> Training Model: PCA_MLP


Epoch 2499, loss: 430.099: : 2500it [03:49, 10.90it/s, train=430.0990, val=432.8955]



>>> Training Model: MLP


Epoch 2499, loss: 433.901: : 2500it [02:41, 15.44it/s, train=433.9008, val=435.3599]


In [35]:
for key, value in cms.items():
    forward_model = value["forward"]
    inverse_model = value["inverse"]
    decoder_model = value["decoder"]
    
    # Forward and inverse outputs
    ut_forward = forward_model(x, t=t, y=y)
    ut_inverse = inverse_model(ut_ref)
    ut_theta = decoder_model(ut_forward,t=t,y=y)
    ut_theta_i=decoder_model(ut_inverse,t=t,y=y)
    
    print(f"Model: {key}")
    
    # Loss and MSE
    loss = einsum(torch.square(ut_theta - ut_ref), 'b c h w -> b').mean()
    loss2=einsum(torch.square(  ut_theta_i - ut_ref), 'b c h w -> b').mean()
    loss3=einsum(torch.square(  ut_theta_i - ut_theta), 'b c h w -> b').mean()
    mse = F.mse_loss(ut_theta, ut_ref)
    print(f"  Loss forward-decoder: {loss}")
    print(f"  Loss inverse-decoder: {loss2}")
    print(f"  Loss forward-inverse: {loss3}")
    print(f"  MSE: {mse}")
    
    cos_sim = cosine_similarity(
        rearrange(ut_theta, 'b c h w -> b (c h w)'),
        rearrange(ut_ref, 'b c h w -> b (c h w)')
    )
    print(f"  Cosine Similarity: {cos_sim.mean()}")

Model: MiniViT
  Loss forward-decoder: 1411.8720703125
  Loss inverse-decoder: 1453.01806640625
  Loss forward-inverse: 10.484301567077637
  MSE: 1.3787813186645508
  Cosine Similarity: 0.5555613040924072
Model: UNet
  Loss forward-decoder: 1273.862548828125
  Loss inverse-decoder: 1213.293212890625
  Loss forward-inverse: 37.902435302734375
  MSE: 1.2440063953399658
  Cosine Similarity: 0.5916528701782227
Model: PCA_MLP
  Loss forward-decoder: 1300.2447509765625
  Loss inverse-decoder: 1235.023193359375
  Loss forward-inverse: 23.82490348815918
  MSE: 1.2697702646255493
  Cosine Similarity: 0.5760877728462219
Model: MLP
  Loss forward-decoder: 1265.468017578125
  Loss inverse-decoder: 1316.6497802734375
  Loss forward-inverse: 11.435264587402344
  MSE: 1.2358086109161377
  Cosine Similarity: 0.5888048410415649


## for-inv-full

In [36]:
fullmodels = {
    "MiniViT": {
        "forward": LatentVecField(MiniVisionTransformer(
                    d_in=1, d_out=16, d_hidden=32, patch_size=4, num_heads=4, img_size=32),16),
        "inverse": MiniVisionTransformer(
                    d_in=1, d_out=16, d_hidden=32, patch_size=4, num_heads=4, img_size=32),
        "decoder":Utdecoder(16,1)
    },
    "UNet": {
        "forward": LatentVecField(MiniUNet(
                    d_in=1, d_out=16, d_hidden=26, kernel_num=3, img_size=32
                ),16),
        "inverse": MiniUNet(
                    d_in=1, d_out=16, d_hidden=26, kernel_num=3, img_size=32
                ),
         "decoder":Utdecoder(16,1)
    },
    "PCA_MLP": {
        "forward": LatentVecField(PCAMLP(
                    d_in=32*32, d_out=16, d_hidden=57, img_size=32, n_components=256
                ), 16),
        "inverse": PCAMLP(
                    d_in=32*32, d_out=16, d_hidden=58, img_size=32, n_components=256
                ),
         "decoder":Utdecoder(16,1)
    },
    "MLP": {
        "forward": LatentVecField(MLPExtractor(
                    d_in=32*32, d_out=14, d_hidden=16, img_size=32
                ), 14),
        "inverse": MLPExtractor(
                    d_in=32*32, d_out=14, d_hidden=16, img_size=32
                ),
         "decoder":Utdecoder(14,1) 
    }
}

In [37]:
runner = FullFlowExperimentRunner(
    models=fullmodels,
    cfg_class=FullCFG,
    path=path,           
    num_epochs=2500,
    batch_size=128,
    device=device,
    eta=0.1,
    lr=1e-3
)

runner.run()


>>> Training Model: MiniViT


Epoch 2499, loss: 594.458: : 2500it [05:20,  7.81it/s, train=594.4583, val=597.6832]



>>> Training Model: UNet


Epoch 2499, loss: 602.158: : 2500it [05:38,  7.38it/s, train=602.1583, val=600.1739]



>>> Training Model: PCA_MLP


Epoch 2499, loss: 592.927: : 2500it [04:06, 10.12it/s, train=592.9275, val=592.6620]



>>> Training Model: MLP


Epoch 2499, loss: 570.944: : 2500it [03:02, 13.69it/s, train=570.9438, val=570.8911]


In [38]:
ams= {
    "MiniViT":  vecfield(fullmodels['MiniViT']['forward'],fullmodels['MiniViT']['decoder']),
    
    "MiniUNet":vecfield(fullmodels['UNet']['forward'],fullmodels['UNet']['decoder']),
    "PCAMLP": vecfield(fullmodels['PCA_MLP']['forward'],fullmodels['PCA_MLP']['decoder']),
    "MLP":vecfield(fullmodels['MLP']['forward'],fullmodels['MLP']['decoder']),
}

In [39]:
for key, value in fullmodels.items():
    forward_model = value["forward"]
    inverse_model = value["inverse"]
    decoder_model = value["decoder"]
    
    # Forward and inverse outputs
    ut_forward = forward_model(x, t=t, y=y)
    ut_inverse = inverse_model(ut_ref)
    ut_theta = decoder_model(ut_forward,t=t,y=y)
    ut_theta_i=decoder_model(ut_inverse,t=t,y=y)
    
    print(f"Model: {key}")
    
    # Loss and MSE
    loss = einsum(torch.square(ut_theta - ut_ref), 'b c h w -> b').mean()
    loss2=einsum(torch.square(  ut_theta_i - ut_ref), 'b c h w -> b').mean()
    loss3=einsum(torch.square(  ut_theta_i - ut_theta), 'b c h w -> b').mean()
    mse = F.mse_loss(ut_theta, ut_ref)
    print(f"  Loss forward-decoder: {loss}")
    print(f"  Loss inverse-decoder: {loss2}")
    print(f"  Loss forward-inverse: {loss3}")
    print(f"  MSE: {mse}")
    
    cos_sim = cosine_similarity(
        rearrange(ut_theta, 'b c h w -> b (c h w)'),
        rearrange(ut_ref, 'b c h w -> b (c h w)')
    )
    print(f"  Cosine Similarity: {cos_sim.mean()}")

Model: MiniViT
  Loss forward-decoder: 1801.9765625
  Loss inverse-decoder: 1797.1920166015625
  Loss forward-inverse: 0.013409857638180256
  MSE: 1.7597427368164062
  Cosine Similarity: 0.5401809215545654
Model: UNet
  Loss forward-decoder: 1808.115234375
  Loss inverse-decoder: 1806.2037353515625
  Loss forward-inverse: 0.0021129099186509848
  MSE: 1.7657376527786255
  Cosine Similarity: 0.53992760181427
Model: PCA_MLP
  Loss forward-decoder: 1762.1171875
  Loss inverse-decoder: 1754.76318359375
  Loss forward-inverse: 0.2825869619846344
  MSE: 1.7208176851272583
  Cosine Similarity: 0.4642002284526825
Model: MLP
  Loss forward-decoder: 1702.256591796875
  Loss inverse-decoder: 1699.387451171875
  Loss forward-inverse: 0.132313072681427
  MSE: 1.6623599529266357
  Cosine Similarity: 0.4932558834552765


In [40]:
for key, value in ams.items():
    
    
    # Forward and inverse outputs
    ut_theta = value(x, t=t, y=y)
    print(f"Model: {key}")
    
    # Loss and MSE
    loss = einsum(torch.square(ut_theta - ut_ref), 'b  c h w -> b').mean()
    mse = F.mse_loss(ut_theta ,ut_ref)
    print(f"  Loss: {loss}")
    print(f"  MSE: {mse}")
    cos_sim = cosine_similarity(rearrange(ut_theta, 'b c h w-> b (c h w)'), rearrange(ut_ref, 'b c h w -> b (c h w)'))
    print(f"  Cosine Similarity: {cos_sim.mean()}")

Model: MiniViT
  Loss: 1801.9765625
  MSE: 1.7597427368164062
  Cosine Similarity: 0.5401809215545654
Model: MiniUNet
  Loss: 1808.115234375
  MSE: 1.7657376527786255
  Cosine Similarity: 0.53992760181427
Model: PCAMLP
  Loss: 1762.1171875
  MSE: 1.7208176851272583
  Cosine Similarity: 0.4642002284526825
Model: MLP
  Loss: 1702.256591796875
  MSE: 1.6623599529266357
  Cosine Similarity: 0.4932558834552765


## inverse path train

In [41]:
inversemodels2 = {
    "MiniViT": {
        "decoder": Utdecoder(16,1),
        "inverse": latentmodels['MiniViT']['inverse']
    },
    "ConvNet": {
        "decoder":  Utdecoder(16,1),
        "inverse": latentmodels['ConvNet']['inverse']
    },
    "PCA_MLP": {
        "decoder": Utdecoder(16,1),
        "inverse": latentmodels['PCA_MLP']['inverse']
    },
    "MLP": {
        "decoder":Utdecoder(14,1),
        "inverse": latentmodels['MLP']['inverse']
    }
}

In [42]:
for key, models_pair in inversemodels2.items():
    decoder_model = models_pair["decoder"]
    inverse_model = models_pair["inverse"]
    
    print(f"\n[{key}]")
    
    print(f"  Decoder model size: {model_size_b(decoder_model) / MiB:.2f} MiB")
    print(f"  Decoder model params: {count_model_params(decoder_model)}")
    
    print(f"  Inverse model size: {model_size_b(inverse_model) / MiB:.2f} MiB")
    print(f"  Inverse model params: {count_model_params(inverse_model)}")
    
    total_params = count_model_params(decoder_model) + count_model_params(inverse_model)
    total_size=(model_size_b(decoder_model)+model_size_b(inverse_model))/ MiB
    print(f"  Total combined params: {total_params}")
    print(f"  Total combined size: {total_size:.2f} MiB")


[MiniViT]
  Decoder model size: 0.07 MiB
  Decoder model params: 17179
  Inverse model size: 0.06 MiB
  Inverse model params: 15824
  Total combined params: 33003
  Total combined size: 0.13 MiB

[ConvNet]
  Decoder model size: 0.07 MiB
  Decoder model params: 17179
  Inverse model size: 0.06 MiB
  Inverse model params: 16396
  Total combined params: 33575
  Total combined size: 0.13 MiB

[PCA_MLP]
  Decoder model size: 0.07 MiB
  Decoder model params: 17179
  Inverse model size: 1.06 MiB
  Inverse model params: 15850
  Total combined params: 33029
  Total combined size: 1.13 MiB

[MLP]
  Decoder model size: 0.06 MiB
  Decoder model params: 15108
  Inverse model size: 0.06 MiB
  Inverse model params: 16638
  Total combined params: 31746
  Total combined size: 0.12 MiB


In [43]:

runner = InverseFlowExperimentRunner2(
  models=inversemodels2,
    cfg_class=InverseutCFG,
    path=path,           
    num_epochs=5000,
    batch_size=128,
    device=device,
    eta=0.1,
    lr=1e-3)
runner.run()


>>> Training Model: MiniViT


Epoch 4999, loss: 1246.655: : 5000it [08:05, 10.29it/s, train=1246.6545, val=1233.6250]



>>> Training Model: ConvNet


Epoch 4999, loss: 1154.952: : 5000it [07:26, 11.19it/s, train=1154.9517, val=1147.8846]



>>> Training Model: PCA_MLP


Epoch 4999, loss: 1173.854: : 5000it [03:16, 25.38it/s, train=1173.8545, val=1174.5945]



>>> Training Model: MLP


Epoch 4999, loss: 1257.962: : 5000it [03:14, 25.74it/s, train=1257.9624, val=1248.4041]


In [44]:
for key, models_pair in inversemodels2.items():
    decoder_model = models_pair["decoder"]
    inverse_model = models_pair["inverse"]
    ut_inverse = inverse_model(ut_ref)
    ut_theta_i=decoder_model(ut_inverse,t=t,y=y)
    loss2=einsum(torch.square(  ut_theta_i - ut_ref), 'b c h w -> b').mean()
    loss3=einsum(torch.square(  ut_theta_i - ut_theta), 'b c h w -> b').mean()
    print(f"  Loss inverse-decoder: {loss2}")
    print(f"  Loss forward-inverse: {loss3}")
    cos_sim = cosine_similarity(
        rearrange(ut_theta_i, 'b c h w -> b (c h w)'),
        rearrange(ut_ref, 'b c h w -> b (c h w)')
    )
    print(f"  Cosine Similarity: {cos_sim.mean()}")

  Loss inverse-decoder: 1248.8275146484375
  Loss forward-inverse: 454.85980224609375
  Cosine Similarity: 0.5927346348762512
  Loss inverse-decoder: 1155.6533203125
  Loss forward-inverse: 532.3026733398438
  Cosine Similarity: 0.6328291893005371
  Loss inverse-decoder: 1176.9337158203125
  Loss forward-inverse: 525.0035400390625
  Cosine Similarity: 0.6239798665046692
  Loss inverse-decoder: 1258.00732421875
  Loss forward-inverse: 452.3415222167969
  Cosine Similarity: 0.5896075963973999


In [45]:
latentmodels2 = {
    "MiniViT": {
        "forward": LatentVecField(MiniVisionTransformer(
                    d_in=1, d_out=16, d_hidden=32, patch_size=4, num_heads=4, img_size=32),16),
        "inverse": MiniVisionTransformer(
                    d_in=1, d_out=16, d_hidden=32, patch_size=4, num_heads=4, img_size=32)
    },
    "ConvNet": {
        "forward": LatentVecField(MiniUNet(
                    d_in=1, d_out=16, d_hidden=26, kernel_num=3, img_size=32
                ),16),
        "inverse": MiniUNet(
                    d_in=1, d_out=16, d_hidden=26, kernel_num=3, img_size=32
                ),
    },
    "PCA_MLP": {
        "forward": LatentVecField(PCAMLP(
                    d_in=32*32, d_out=16, d_hidden=57, img_size=32, n_components=256
                ), 16),
        "inverse": PCAMLP(
                    d_in=32*32, d_out=16, d_hidden=58, img_size=32, n_components=256
                ),
    },
    "MLP": {
        "forward": LatentVecField(MLPExtractor(
                    d_in=32*32, d_out=14, d_hidden=16, img_size=32
                ), 14),
        "inverse": MLPExtractor(
                    d_in=32*32, d_out=14, d_hidden=16, img_size=32
                )
    }
}

In [46]:
for key, models_pair in latentmodels2.items():
    forward_model = models_pair["forward"]
    inverse_model = models_pair["inverse"]
    
    print(f"\n[{key}]")
    
    print(f"  Forward model size: {model_size_b(forward_model) / MiB:.2f} MiB")
    print(f"  Forward model params: {count_model_params(forward_model)}")
    
    print(f"  Inverse model size: {model_size_b(inverse_model) / MiB:.2f} MiB")
    print(f"  Inverse model params: {count_model_params(inverse_model)}")
    
    total_params = count_model_params(forward_model) + count_model_params(inverse_model)
    total_size=(model_size_b(forward_model)+model_size_b(inverse_model))/ MiB
    print(f"  Total combined params: {total_params}")
    print(f"  Total combined size: {total_size:.2f} MiB")


[MiniViT]
  Forward model size: 0.06 MiB
  Forward model params: 16008
  Inverse model size: 0.06 MiB
  Inverse model params: 15824
  Total combined params: 31832
  Total combined size: 0.13 MiB

[ConvNet]
  Forward model size: 0.06 MiB
  Forward model params: 16580
  Inverse model size: 0.06 MiB
  Inverse model params: 16396
  Total combined params: 32976
  Total combined size: 0.13 MiB

[PCA_MLP]
  Forward model size: 1.06 MiB
  Forward model params: 15761
  Inverse model size: 1.06 MiB
  Inverse model params: 15850
  Total combined params: 31611
  Total combined size: 2.13 MiB

[MLP]
  Forward model size: 0.06 MiB
  Forward model params: 16799
  Inverse model size: 0.06 MiB
  Inverse model params: 16638
  Total combined params: 33437
  Total combined size: 0.13 MiB


In [47]:
runner = LatentFlowExperimentRunner2(
    models=latentmodels2,
    cfg_class=LatentCFG,
    path=path,           
    num_epochs=5000,
    batch_size=128,
    device=device,
    eta=0.1,
    lr=1e-3
)

runner.run()


>>> Training Model: MiniViT


Epoch 4999, loss: 580.181: : 5000it [07:48, 10.66it/s, train=580.1812, val=577.0331]   



>>> Training Model: ConvNet


Epoch 4999, loss: 439.022: : 5000it [07:11, 11.58it/s, train=439.0215, val=457.5040]   



>>> Training Model: PCA_MLP


Epoch 4999, loss: 10.428: : 5000it [02:09, 38.49it/s, train=10.4284, val=9.7806] 



>>> Training Model: MLP


Epoch 4999, loss: 7.189: : 5000it [02:08, 39.06it/s, train=7.1894, val=7.6864]   


In [48]:
for key, models_pair in latentmodels2.items():
    forward_model = models_pair["forward"]
    inverse_model = models_pair["inverse"]

    # Forward and inverse outputs
    ut_forward = forward_model(x, t, y)
    ut_inverse = inverse_model(x)

    print(f"Model: {key}")
    
    # Loss and MSE
    loss = einsum(torch.square(ut_forward - ut_inverse), 'b seq d -> b').mean()
    mse = F.mse_loss(ut_forward, ut_inverse)
    print(f"  Loss: {loss}")
    print(f"  MSE: {mse}")

    # Reshape for probability-based metrics
    ut_forward = rearrange(ut_forward, 'b seq d -> b (seq d)')
    ut_inverse = rearrange(ut_inverse, 'b seq d -> b (seq d)')
    ut_forward_prob = torch.softmax(ut_forward, dim=-1)
    ut_inverse_prob = torch.softmax(ut_inverse, dim=-1)

    # KL Divergence
    kl_div_forward = torch.sum(ut_forward_prob * torch.log(ut_forward_prob / (ut_inverse_prob + 1e-8)), dim=-1)
    kl_div_inverse = torch.sum(ut_inverse_prob * torch.log(ut_inverse_prob / (ut_forward_prob + 1e-8)), dim=-1)
    print(f"  KL Divergence (forward): {kl_div_forward.mean()}")
    print(f"  KL Divergence (inverse): {kl_div_inverse.mean()}")

    # Wasserstein Distance
    wasserstein_distance = pairwise_distance(ut_forward, ut_inverse, p=2).mean()
    print(f"  Wasserstein Distance: {wasserstein_distance}")

    # JS Divergence
    m = 0.5 * (ut_forward_prob + ut_inverse_prob)
    kl_forward = torch.sum(ut_forward_prob * torch.log(ut_forward_prob / (m + 1e-8)), dim=-1)
    kl_inverse = torch.sum(ut_inverse_prob * torch.log(ut_inverse_prob / (m + 1e-8)), dim=-1)
    js_divergence = 0.5 * (kl_forward + kl_inverse)
    print(f"  JS Divergence: {js_divergence.mean()}")

    # Cosine Similarity
    cos_sim = cosine_similarity(ut_forward, ut_inverse, dim=-1)
    print(f"  Cosine Similarity: {cos_sim.mean()}")

    # Bhattacharyya Distance
    bc_coefficient = torch.sum(torch.sqrt(ut_forward_prob * ut_inverse_prob), dim=-1)
    bhattacharyya_distance = -torch.log(bc_coefficient + 1e-8)
    print(f"  Bhattacharyya Distance: {bhattacharyya_distance.mean()}")

Model: MiniViT
  Loss: 1851.7225341796875
  MSE: 1.7805026769638062
  KL Divergence (forward): 0.9594736695289612
  KL Divergence (inverse): 0.8990741968154907
  Wasserstein Distance: 41.081878662109375
  JS Divergence: 0.17121915519237518
  Cosine Similarity: 0.5608558654785156
  Bhattacharyya Distance: 0.22454997897148132
Model: ConvNet
  Loss: 1802.8851318359375
  MSE: 0.44015753269195557
  KL Divergence (forward): 0.22358646988868713
  KL Divergence (inverse): 0.21997492015361786
  Wasserstein Distance: 41.47486114501953
  JS Divergence: 0.05155692622065544
  Cosine Similarity: -0.21052326261997223
  Bhattacharyya Distance: 0.0550660640001297
Model: PCA_MLP
  Loss: 7.848595142364502
  MSE: 0.49053725600242615
  KL Divergence (forward): 0.2342039942741394
  KL Divergence (inverse): 0.2343112975358963
  Wasserstein Distance: 2.6846725940704346
  JS Divergence: 0.05494900047779083
  Cosine Similarity: 0.46918824315071106
  Bhattacharyya Distance: 0.058995701372623444
Model: MLP
  Loss

In [49]:
mms2 = {
    "MiniViT":  vecfield(latentmodels2['MiniViT']['forward'],inversemodels2['MiniViT']['decoder']),
    
    "MiniUNet":vecfield(latentmodels2['ConvNet']['forward'],inversemodels2['ConvNet']['decoder']),
    "PCAMLP": vecfield(latentmodels2['PCA_MLP']['forward'],inversemodels2['PCA_MLP']['decoder']),
    "MLP":vecfield(latentmodels2['MLP']['forward'],inversemodels2['MLP']['decoder']),
}

In [50]:
for key, value in mms2.items():
    
    # Forward and inverse outputs
    ut_theta = value(x, t=t, y=y)
    print(f"Model: {key}")
    
    # Loss and MSE
    loss = einsum(torch.square(ut_theta - ut_ref), 'b  c h w -> b').mean()
    mse = F.mse_loss(ut_theta ,ut_ref)
    print(f"  Loss: {loss}")
    print(f"  MSE: {mse}")
    cos_sim = cosine_similarity(rearrange(ut_theta, 'b c h w-> b (c h w)'), rearrange(ut_ref, 'b c h w -> b (c h w)'))
    print(f"  Cosine Similarity: {cos_sim.mean()}")

Model: MiniViT
  Loss: 532593.6875
  MSE: 520.1110229492188
  Cosine Similarity: -0.3486294746398926
Model: MiniUNet
  Loss: 34469.02734375
  MSE: 33.66115951538086
  Cosine Similarity: -0.0961483046412468
Model: PCAMLP
  Loss: 18777.224609375
  MSE: 18.337133407592773
  Cosine Similarity: -0.03154664486646652
Model: MLP
  Loss: 1664.3912353515625
  MSE: 1.6253820657730103
  Cosine Similarity: 0.4582255780696869


In [51]:
runner = FlowExperimentRunner(
    models=mms2,
    cfg_class=CFG,       
    path=path,
    eta=0.1,
    num_epochs=2500,
    batch_size=250,
    lr=1e-3,
    device=device
    
)
runner.run()


>>> Training Model: MiniViT


Epoch 2499, loss: 1373.613: : 2500it [05:27,  7.64it/s, train=1373.6133, val=1381.1670]



>>> Training Model: MiniUNet


Epoch 2499, loss: 1364.942: : 2500it [05:32,  7.51it/s, train=1364.9420, val=1360.2684]



>>> Training Model: PCAMLP


Epoch 2499, loss: 1380.919: : 2500it [03:24, 12.24it/s, train=1380.9191, val=1366.8558]



>>> Training Model: MLP


Epoch 2499, loss: 1163.641: : 2500it [03:17, 12.67it/s, train=1163.6412, val=1154.4805]


In [52]:
for key, value in mms2.items():
    
    # Forward and inverse outputs
    ut_theta = value(x, t=t, y=y)
    print(f"Model: {key}")
    
    # Loss and MSE
    loss = einsum(torch.square(ut_theta - ut_ref), 'b  c h w -> b').mean()
    mse = F.mse_loss(ut_theta ,ut_ref)
    print(f"  Loss: {loss}")
    print(f"  MSE: {mse}")
    cos_sim = cosine_similarity(rearrange(ut_theta, 'b c h w-> b (c h w)'), rearrange(ut_ref, 'b c h w -> b (c h w)'))
    print(f"  Cosine Similarity: {cos_sim.mean()}")

Model: MiniViT
  Loss: 1365.9854736328125
  MSE: 1.333970308303833
  Cosine Similarity: 0.5399560928344727
Model: MiniUNet
  Loss: 1365.9605712890625
  MSE: 1.333945870399475
  Cosine Similarity: 0.5399670004844666
Model: PCAMLP
  Loss: 1362.826904296875
  MSE: 1.330885648727417
  Cosine Similarity: 0.5415140390396118
Model: MLP
  Loss: 1157.52490234375
  MSE: 1.1303954124450684
  Cosine Similarity: 0.6318214535713196


In [53]:
import os

In [60]:
def save_complex_models(nested_dict, save_dir="saved_models"):
    os.makedirs(save_dir, exist_ok=True)

    for top_key, sub_dict in nested_dict.items():
        model_folder = os.path.join(save_dir, top_key)
        os.makedirs(model_folder, exist_ok=True)

        if isinstance(sub_dict, dict):
           
            for name, model in sub_dict.items():
                if isinstance(model, dict):
                   
                    for subname, submodel in model.items():
                        path = os.path.join(model_folder, f"{name}_{subname}.pth")
                        torch.save(submodel.state_dict(), path)
                else:
             
                    path = os.path.join(model_folder, f"{name}.pth")
                    torch.save(model.state_dict(), path)
        else:
      
            path = os.path.join(save_dir, f"{top_key}.pth")
            torch.save(sub_dict.state_dict(), path)

    print(f"✔️ Models saved to {save_dir}")

In [55]:
def load_complex_models(nested_dict, save_dir="saved_models"):
    for top_key, sub_dict in nested_dict.items():
        model_folder = os.path.join(save_dir, top_key)

        if isinstance(sub_dict, dict):
            for name, model in sub_dict.items():
                if isinstance(model, dict):
                    for subname, submodel in model.items():
                        path = os.path.join(model_folder, f"{name}_{subname}.pt")
                        if os.path.exists(path):
                            submodel.load_state_dict(torch.load(path))
                else:
                    path = os.path.join(model_folder, f"{name}.pt")
                    if os.path.exists(path):
                        model.load_state_dict(torch.load(path))
        else:
            path = os.path.join(save_dir, f"{top_key}.pt")
            if os.path.exists(path):
                sub_dict.load_state_dict(torch.load(path))

    print(f"✔️ Models loaded from {save_dir}")

In [ ]:
inversemodels2
latentmodels2
mms2

In [61]:
models_group = {
    "modelo": modelo,
    "models_full": models_full,
    "latentmodels": latentmodels,
    "inversemodels": inversemodels,
    "mms": mms,
    "cms": cms,
    "inversemodels2":inversemodels2,
    "latentmodels2":latentmodels2,
      "mms2": mms2,
    
    
}


save_complex_models(models_group, save_dir="saved_vecfield_models")

✔️ Models saved to saved_vecfield_models


In [64]:
def save_complex_models(nested_dict, save_dir="saved_models"):
    os.makedirs(save_dir, exist_ok=True)

    for top_key, sub_dict in nested_dict.items():
        model_folder = os.path.join(save_dir, top_key)
        os.makedirs(model_folder, exist_ok=True)

        if isinstance(sub_dict, dict):
            for name, model in sub_dict.items():
               
                if isinstance(model, dict):
                    for subname, submodel in model.items():
                        path = os.path.join(model_folder, f"{name}_{subname}.pth")
                        print(f"Saving {top_key} / {name}_{subname} to {path}")
                        torch.save(submodel.state_dict(), path)
                else:
                    path = os.path.join(model_folder, f"{name}.pth")
                    print(f"Saving {top_key} / {name} to {path}")
                    torch.save(model.state_dict(), path)
        else:
          
            path = os.path.join(save_dir, f"{top_key}.pth")
            print(f"Saving {top_key} to {path}")
            torch.save(sub_dict.state_dict(), path)

    print(f"✔️ Models saved to {save_dir}")

In [65]:
save_complex_models(models_group, save_dir="saved_models")

Saving modelo / MiniViT to saved_models/modelo/MiniViT.pth
Saving modelo / MiniUNet to saved_models/modelo/MiniUNet.pth
Saving modelo / PCAMLP to saved_models/modelo/PCAMLP.pth
Saving modelo / MLP to saved_models/modelo/MLP.pth
Saving models_full / MiniViT to saved_models/models_full/MiniViT.pth
Saving models_full / MiniUNet to saved_models/models_full/MiniUNet.pth
Saving models_full / PCAMLP to saved_models/models_full/PCAMLP.pth
Saving models_full / MLP to saved_models/models_full/MLP.pth
Saving latentmodels / MiniViT_forward to saved_models/latentmodels/MiniViT_forward.pth
Saving latentmodels / MiniViT_inverse to saved_models/latentmodels/MiniViT_inverse.pth
Saving latentmodels / ConvNet_forward to saved_models/latentmodels/ConvNet_forward.pth
Saving latentmodels / ConvNet_inverse to saved_models/latentmodels/ConvNet_inverse.pth
Saving latentmodels / PCA_MLP_forward to saved_models/latentmodels/PCA_MLP_forward.pth
Saving latentmodels / PCA_MLP_inverse to saved_models/latentmodels/PC

# expr

In [57]:
z, y = path.p_data.sample(2000) 
t = torch.rand(2000,1,1,1).to(z)
x = path.sample_conditional_path(z,t) 
ut_ref = path.conditional_vector_field(x,z,t) 

In [58]:
for key, value in modelo.items():
    
    # Forward and inverse outputs
    ut_theta = value(x, t=t, y=y)
    print(f"Model: {key}")
    
    # Loss and MSE
    loss = einsum(torch.square(ut_theta - ut_ref), 'b  c h w -> b').mean()
    mse = F.mse_loss(ut_theta ,ut_ref)
    print(f"  Loss: {loss}")
    print(f"  MSE: {mse}")
    cos_sim = cosine_similarity(rearrange(ut_theta, 'b c h w-> b (c h w)'), rearrange(ut_ref, 'b c h w -> b (c h w)'))
    print(f"  Cosine Similarity: {cos_sim.mean()}")

Model: MiniViT
  Loss: 1158.0556640625
  MSE: 1.1309138536453247
  Cosine Similarity: 0.6302759051322937
Model: MiniUNet
  Loss: 1197.665771484375
  MSE: 1.16959547996521
  Cosine Similarity: 0.6136642694473267
Model: PCAMLP
  Loss: 1193.004638671875
  MSE: 1.165043592453003
  Cosine Similarity: 0.6158406734466553
Model: MLP
  Loss: 1243.132568359375
  MSE: 1.213996410369873
  Cosine Similarity: 0.5945520401000977


In [59]:
for key, value in models_full.items():
    
    # Forward and inverse outputs
    ut_theta = value(x, t=t, y=y)
    print(f"Model: {key}")
    
    # Loss and MSE
    loss = einsum(torch.square(ut_theta - ut_ref), 'b  c h w -> b').mean()
    mse = F.mse_loss(ut_theta ,ut_ref)
    print(f"  Loss: {loss}")
    print(f"  MSE: {mse}")
    cos_sim = cosine_similarity(rearrange(ut_theta, 'b c h w-> b (c h w)'), rearrange(ut_ref, 'b c h w -> b (c h w)'))
    print(f"  Cosine Similarity: {cos_sim.mean()}")

Model: MiniViT
  Loss: 1203.028564453125
  MSE: 1.1748327016830444
  Cosine Similarity: 0.6114094257354736
Model: MiniUNet
  Loss: 1203.220947265625
  MSE: 1.1750205755233765
  Cosine Similarity: 0.6117534041404724
Model: PCAMLP
  Loss: 1163.03515625
  MSE: 1.1357765197753906
  Cosine Similarity: 0.628250241279602
Model: MLP
  Loss: 1161.2552490234375
  MSE: 1.1340383291244507
  Cosine Similarity: 0.6289087533950806


In [ ]:
 latentmodels['MiniViT']['inverse']

In [66]:
print(latentmodels['MiniViT']['inverse']  is inversemodels['MiniViT']['inverse'])

True
